# Teach an LLM to do additions

The goal of this project is to teach an LLM to do additions, playing only with two parts:
* the tokenizer
* the positional embedding

Both the model and the dataset are fixed.

You are allowed to tune the hyperparameters, but this is not the main goal. Depending on the quality of your tokenizer and positional embedding, you may change the number of bits. The initial value of 3 is very small.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

import random
import math
import re
import time

In [2]:
number_bits = 9

dataset_size = 64_000
train_proportion = 0.9

log_interval = 200
batch_size = 64
epochs = 4
learning_rate = 8e-4

## Step 1: Construct a tokenizer

In [3]:
pad_token="[PAD]"
eos_token="[EOS]"

### Baseline: character-level tokenizer

In [4]:
class character_level_tokenizer:
    """
    character-level
    """
    def __init__(self):
        self.vocab = [str(x) for x in range(10)] + ["+", "="] + [pad_token, eos_token]
        self.token_to_id = {v : k for k, v in enumerate(self.vocab)}
        self.id_to_token = {k : v for k, v in enumerate(self.vocab)}
        self.ntokens = len(self.vocab)
        self.pattern = f"[^{re.escape(''.join(self.vocab))}]"

    def clean(self, text):
        """
        removes all characters not in the vocabulary
        """
        out = re.sub(self.pattern, "", text)
        return out

    def pre_tokenization(self, text):
        """
        character-level
        """
        return [c for c in text]

    def encode(self, text):
        text_list = self.pre_tokenization(self.clean(text))
        return [self.token_to_id[c] for c in text_list]

    def decode(self, token_list):
        return "".join([self.id_to_token[x] for x in token_list])

In [5]:
tokenizer = character_level_tokenizer()
ntokens = tokenizer.ntokens
ntokens

14

In [6]:
prompt = "12 + 42 ="
inputs = tokenizer.encode(prompt)
inputs, tokenizer.decode(inputs)

([1, 2, 10, 4, 2, 11], '12+42=')

# Implement your tokenizer here!

You can do anything (as long as you do not compute the addition!).
Some ideas:
* reversing numbers left to right
* arranging by groups (of, 2, 3,...)
* aligning numbers

In [7]:
import re

class DigitTokenizer:
    """
    Tokenizer that aligns and flips numbers for addition.
      - group_size allows to group digit before giving them to the model.
    """
    def __init__(self, group_size=1):
        self.group_size = group_size

        number_pairs = [
            f"{str(i).zfill(group_size)},{str(j).zfill(group_size)}"
            for i in range(10 ** group_size)
            for j in range(10 ** group_size)
        ]
        self.vocab = number_pairs + ["+", "="] + [pad_token, eos_token]

        self.token_to_id = {v: k for k, v in enumerate(self.vocab)}
        self.id_to_token = {k: v for k, v in enumerate(self.vocab)}
        self.ntokens = len(self.vocab)

    def clean(self, text):
        """Removes all characters not in the vocabulary."""
        return re.sub(r"[^0-9+=]", "", text)

    def align_and_flip(self, left, right):
        """
        Aligns numbers from right to left, grouping into chunks of `group_size`.
        """
        max_len = max(len(left), len(right))
        left = left.zfill(max_len)[::-1]   # Reverse left number
        right = right.zfill(max_len)[::-1] # Reverse right number

        left_groups = [
            left[i : i + self.group_size][::-1].zfill(self.group_size)
            for i in range(0, max_len, self.group_size)
        ]
        right_groups = [
            right[i : i + self.group_size][::-1].zfill(self.group_size)
            for i in range(0, max_len, self.group_size)
        ]

        return list(zip(left_groups, right_groups))

    def pre_tokenization(self, text):
        """Splits text into tokens and aligns numbers.
        """
        tokens = []
        parts = re.split(r"(\+|=)", text)

        if len(parts) >= 3: # If we need to encode an addition
            left, operator, right = parts[:3] # get the two numbers
            aligned_pairs = self.align_and_flip(left, right)
            grouped_tokens = [",".join(pair) for pair in aligned_pairs]
            tokens.extend(grouped_tokens)
            tokens.append(operator)
            if "=" in text:
                tokens.append("=")
        else: # else we encode the answer as if it was result + 0
            left = parts[0]
            right = "0" * len(left)
            aligned_pairs = self.align_and_flip(left, right)
            grouped_tokens = [",".join(pair) for pair in aligned_pairs]
            tokens.extend(grouped_tokens)

        return tokens

    def encode(self, text):
        """Converts text into a sequence of token IDs."""
        text_list = self.pre_tokenization(self.clean(text))
        return [self.token_to_id[token] for token in text_list]

    def decode(self, token_list):
        """
        Converts a sequence of token IDs back into text, properly formatted.
        """
        tokens = [self.id_to_token[x] for x in token_list if x in self.id_to_token]

        first_number = ""
        second_number = ""
        result_number = ""
        output = ""
        plus_encountered = False
        equal_encountered = False

        for token in tokens:
            if token == pad_token:
                continue
            if token == eos_token:
                break

            if token == "+":
                plus_encountered = True

                first_str = str(int(first_number)) if first_number else "0" # remove useless 0
                second_str = str(int(second_number)) if second_number else "0" # remove useless 0
                output = first_str + token + second_str
            elif token == "=":
                output += token
                equal_encountered = True

            else:
                if equal_encountered:
                  result_str = token.split(",")[0] # get result
                  result_number = result_str + result_number
                else:
                    # Token format: "xx,yy"
                    first_part, second_part = token.split(",")

                    first_number = first_part + first_number
                    second_number = second_part + second_number

        output = output + result_number if result_number else output

        if not plus_encountered and first_number:
            output = str(int(first_number))

        return output

tokenizer = DigitTokenizer(group_size=1)

# Case 1: Addition expression.
text = "1+242="
encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded)
print(f"Original: {text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")
print()

# Case 2: Single number
text2 = "1242"
encoded2 = tokenizer.encode(text2)
decoded2 = tokenizer.decode(encoded2)
print(f"Original: {text2}")
print(f"Encoded: {encoded2}")
print(f"Decoded: {decoded2}")


Original: 1+242=
Encoded: [12, 4, 2, 100, 101]
Decoded: 1+242=

Original: 1242
Encoded: [20, 40, 20, 10]
Decoded: 1242


In [8]:
tokenizer = DigitTokenizer()
ntokens = tokenizer.ntokens

## Step 2: Create a dataset for arithmetic operations

In [9]:
def sample_datapoint(number_bits = 3):
    """
    returns a string containing two random numbers on `number_bits` many bits and their sum.
    """
    a_list = [random.randint(0, 9) for _ in range(number_bits)]
    b_list = [random.randint(0, 9) for _ in range(number_bits)]
    a_int = int("".join([str(x) for x in a_list]))
    b_int = int("".join([str(x) for x in b_list]))
    sum_int = a_int + b_int
    return (str(a_int) + "+" + str(b_int) + "=", str(sum_int))

sample_datapoint(3)

('962+614=', '1576')

In [10]:
data = []
for _ in range(dataset_size):
    data.append(sample_datapoint(number_bits))
data[:4]

[('880911803+176033313=', '1056945116'),
 ('51290501+815167545=', '866458046'),
 ('610819779+504179439=', '1114999218'),
 ('444741203+366233260=', '810974463')]

In [11]:
data_train = data[: int(train_proportion * dataset_size)]
data_test = data[int(train_proportion * dataset_size):]

len(data_train),len(data_test)

(57600, 6400)

## Step 3: Construct a model

### Basline: the classical Positional Embedding

In [12]:
class PositionalEmbedding(nn.Module):
    r"""Inject some information about the relative or absolute position of the tokens in the sequence.
        The positional encodings have the same dimension as the embeddings, so that the two can be summed.
        Here, we use sine and cosine functions of different frequencies.
    .. math:
        \text{PosEmbedder}(pos, 2i) = sin(pos/10000^(2i/d_model))
        \text{PosEmbedder}(pos, 2i+1) = cos(pos/10000^(2i/d_model))
        \text{where pos is the word position and i is the embed idx)
    Args:
        d_model: the embed dim (required).
        dropout: the dropout value (default=0.1).
        max_len: the max. length of the incoming sequence (default=5000).
    """

    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEmbedding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        r"""Inputs of forward function
        Args:
            x: the sequence fed to the positional encoder model (required).
        Shape:
            x: [sequence length, batch size, embed dim]
            output: [sequence length, batch size, embed dim]
        """

        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

# Implement your positional embedding here!

You can do anything. Some ideas:
* RoPE
* (randomised) FIRE
* Abacus

**!!! IMPORTANT !!!** This model of Transformers is "input first", meaning that an input is a tensor with shape
(length_prompts, batch_size)

In [13]:
class RoPEPositionalEmbedding(nn.Module):
    def __init__(self, d_model, base=10000):
        super().__init__()
        self.d_model = d_model
        self.base = base

        self.theta = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))

    def forward(self, x):

        seq_len = x.shape[1]
        position = torch.arange(seq_len, dtype=torch.float).unsqueeze(1)

        angles = position * self.theta
        angles = angles.to(device)
        cos, sin = torch.cos(angles), torch.sin(angles)

        x_even, x_odd = x[..., 0::2], x[..., 1::2]
        x_rotated = torch.cat([x_even * cos - x_odd * sin, x_even * sin + x_odd * cos], dim=-1)

        return x_rotated


In [14]:
class TransformerModel(nn.Transformer):
    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):
        super(TransformerModel, self).__init__(d_model=ninp,
                                               nhead=nhead,
                                               dim_feedforward=nhid,
                                               num_encoder_layers=nlayers)
        self.input_emb = nn.Embedding(ntoken, ninp)
        self.pos_encoder = PositionalEmbedding(ninp, dropout) # our implementation of the tokenizer it almost equivalent to using an abacus positional embedding
        self.decoder = nn.Linear(ninp, ntoken)

        self.ninp = ninp
        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        nn.init.uniform_(self.input_emb.weight, -initrange, initrange)
        nn.init.zeros_(self.decoder.bias)
        nn.init.uniform_(self.decoder.weight, -initrange, initrange)

    def _generate_square_subsequent_mask(self, sz):
        return torch.log(torch.tril(torch.ones(sz,sz)))

    def forward(self, src):
        mask = self._generate_square_subsequent_mask(len(src)).to(device)
        self.src_mask = mask

        src = self.input_emb(src) * math.sqrt(self.ninp)
        src = self.pos_encoder(src)
        output_enc = self.encoder(src, mask=self.src_mask)
        output_dec = self.decoder(output_enc)
        return F.log_softmax(output_dec, dim=-1), output_enc

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Please do not change these parameters!

In [16]:
model = TransformerModel(ntoken = ntokens,
                         ninp = 128,
                         nhead = 16,
                         nhid = 64,
                         nlayers = 8)
model.to(device)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


TransformerModel(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-7): 8 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): Linear(in_features=128, out_features=104, bias=True)
  (input_emb): Embedding(104, 128)
  (pos_encoder): PositionalEmbedding(
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [17]:
def generate(model, prompts, new_tokens = 5):
    input_tensor = prompts # (length_prompts, batch_size)
    input_tensor = input_tensor.to(device)
    for _ in range(new_tokens):
        output, _ = model(input_tensor) # (length_prompts, batch_size, ntokens)
        last_output = output[-1,:,:] # (batch_size, ntokens)
        token = torch.argmax(last_output, -1).view((1,-1)) # (1, batch_size)
        input_tensor = torch.cat((input_tensor, token), 0)
    return input_tensor

In [18]:
model.eval()

prompt = "2+3="
prompt_tensor = torch.tensor(tokenizer.encode(prompt)).view((-1,1))
output = generate(model, prompt_tensor).view((1,-1))
output, tokenizer.decode(output.tolist()[0])

(tensor([[ 23, 100, 101,   8,   8,   8,  26,  19]], device='cuda:0'),
 '2+3=12000')

In [19]:
def pad(token_list, type_list = "prompts"):
    max_length = max([len(x) for x in token_list])
    out = []
    for x in token_list:
        if type_list == "prompts":
            out.append([tokenizer.token_to_id[pad_token]] * (max_length - len(x)) + x)
        if type_list == "answers":
            out.append(x + [tokenizer.token_to_id[eos_token]] + [tokenizer.token_to_id[pad_token]] * (max_length - len(x)))
    return out, max_length

In [20]:
prompts = [tokenizer.encode("1+1="), tokenizer.encode("21+35=")]
answers = [tokenizer.encode("2"), tokenizer.encode("56")]
padded_prompts, _ = pad(prompts, "prompts")
padded_answers, _ = pad(answers, "answers")
padded_prompts, padded_answers
[tokenizer.decode(p) for p in padded_prompts], [tokenizer.decode(p) for p in padded_answers]

(['1+1=', '21+35='], ['2', '56'])

In [21]:
def get_batch(split, i):
    data = data_train if split == 'train' else data_test
    prompts = [tokenizer.encode(data[i][0]) for i in range(i, i + batch_size)]
    padded_prompts, length_prompts = pad(prompts, "prompts")
    answers = [tokenizer.encode(data[i][1]) for i in range(i, i + batch_size)]
    padded_answers, length_answers = pad(answers, "answers")
    X = torch.stack([torch.tensor(x) for x in padded_prompts], 1)
    Y = torch.stack([torch.tensor(x) for x in padded_answers], 1)
    return X, Y, length_prompts, length_answers

In [22]:
X, Y, length_prompts, length_answers = get_batch("train", 243)
X.shape, Y.shape, length_prompts, length_answers

(torch.Size([11, 64]), torch.Size([11, 64]), 11, 10)

## Step 4: Evaluate

In [23]:
def evaluate():
    # Turn on evaluation mode disables dropout.
    model.eval()
    correct = 0.
    with torch.no_grad():
        for batch, i in enumerate(range(0, len(data_test) - 1, batch_size)):
            prompts, target_answers, length_prompts, length_answers = get_batch("test", i)
            prompts = prompts.to(device) # (length_prompts, batch_size)
            target_answers = target_answers.to(device) # (length_answers + 1, batch_size)
            output = generate(model, prompts, length_answers + 1) # (length_prompts + length_answers + 1, batch_size)
            answers_tokens = output[length_prompts:, :] # (length_answers + 1, batch_size), contains tokens
            equality_test = answers_tokens == target_answers # (length_answers + 1, batch_size), contains boolean values
            correct += torch.all(equality_test, axis=0).float().sum()
        accuracy = correct / len(data_test)
    return accuracy.item()

In [24]:
evaluate()

0.0

## Step 4: Train the model

In [25]:
def train_epoch():
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    total_loss = 0.
    start_time = time.time()
    for batch, i in enumerate(range(0, len(data_train) - 1, batch_size)):
        prompts, target_answers, length_prompts, length_answers = get_batch("train", i)
        prompts = prompts.to(device) # (length_prompts, batch_size)
        target_answers = target_answers.to(device) # (length_answers, batch_size)
        input_tensor = torch.cat((prompts, target_answers), 0) # (length_prompts + length_answers, batch_size)
        model.zero_grad()
        output, _ = model(input_tensor) # (length_prompts + length_answers, batch_size, ntokens)
        output_answers = output[length_prompts-1:-1,:,:].reshape(-1, ntokens) # (length_answers * batch_size, ntokens)
        target_answers = target_answers.view(-1)
        loss = F.cross_entropy(output_answers, target_answers)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch % log_interval == 0 and batch > 0:
            cur_loss = total_loss / log_interval
            elapsed = time.time() - start_time
            print('| {:5d}/{:5d} batches | ms/batch {:5.2f} | loss {:5.2f} | perplexity {:8.2f}'.format(batch, len(data_train) // batch_size,
                                                                                                        elapsed * 1000 / log_interval, cur_loss, math.exp(cur_loss)))
            total_loss = 0
            start_time = time.time()

def train():
    best_test_accuracy = None
    test_accuracy = evaluate()
    print('-' * 89)
    print('| initialisation | test accuracy {:5.2f}'.format(test_accuracy))
    print('-' * 89)
    for epoch in range(1, epochs+1):
        epoch_start_time = time.time()
        train_epoch()
        test_accuracy = evaluate()
        print('-' * 89)
        print('| end of epoch {:3d} | time: {:5.2f}s | test accuracy {:5.2f}'.format(epoch, (time.time() - epoch_start_time), test_accuracy))
        print('-' * 89)
        # Save the model if the test accuracy is the best we've seen so far.
        if not best_test_accuracy or test_accuracy < best_test_accuracy:
            with open("arithmetic.pt", 'wb') as f:
                torch.save(model, f)
            best_test_accuracy = test_accuracy

In [26]:
train()

-----------------------------------------------------------------------------------------
| initialisation | test accuracy  0.00
-----------------------------------------------------------------------------------------
|   200/  900 batches | ms/batch 35.09 | loss  2.29 | perplexity     9.86
|   400/  900 batches | ms/batch 27.99 | loss  1.78 | perplexity     5.91
|   600/  900 batches | ms/batch 23.29 | loss  1.42 | perplexity     4.14
|   800/  900 batches | ms/batch 22.31 | loss  1.00 | perplexity     2.73
-----------------------------------------------------------------------------------------
| end of epoch   1 | time: 35.34s | test accuracy  0.41
-----------------------------------------------------------------------------------------
|   200/  900 batches | ms/batch 22.33 | loss  0.49 | perplexity     1.64
|   400/  900 batches | ms/batch 20.15 | loss  0.31 | perplexity     1.36
|   600/  900 batches | ms/batch 22.43 | loss  0.20 | perplexity     1.22
|   800/  900 batches | ms/

In [27]:
model.eval()

for i in range(20):
    prompt, answers = data_test[i]
    prompt_tensor = torch.tensor(tokenizer.encode(prompt)).view((-1,1))
    output = generate(model, prompt_tensor, len(answers)).view((1,-1))
    print(tokenizer.decode(output.tolist()[0]) + "\t actual result: " + answers)

653462347+736097802=1389560149	 actual result: 1389560149
623581799+658031960=1281613759	 actual result: 1281613759
483238772+543454947=1026693719	 actual result: 1026693719
775902289+556358935=1332261224	 actual result: 1332261224
955687900+445793084=1401480984	 actual result: 1401480984
259327697+722212260=981539957	 actual result: 981539957
407588553+172172647=579761200	 actual result: 579761200
857732684+391439163=1249171847	 actual result: 1249171847
342319394+629503172=971822566	 actual result: 971822566
461408226+27602436=489010662	 actual result: 489010662
976553170+941225514=1917778684	 actual result: 1917778684
421834233+747232769=1169067002	 actual result: 1169067002
62719817+76034581=387554398	 actual result: 138754398
525997168+702082654=1228079822	 actual result: 1228079822
227720269+666830415=894550684	 actual result: 894550684
110291571+148102480=258394051	 actual result: 258394051
522756417+750915380=1273671797	 actual result: 1273671797
142742396+375746995=518489391	 

## Probing

This is just for fun...

In [28]:
import numpy as np

train_size = 1000
test_size = 100

model.eval()

def data_probing(size):
    X = []
    y = np.zeros(size)
    for i in range(size):
        input = torch.tensor(tokenizer.encode(data[i][0])).view((-1, 1)).to(device)
        _, output = model(input)
        output = output[-1,:,:].flatten()
        # determine whether there was a carry in the result:
        carry = len(data[i][1]) > len(data[i][0]) / 2
        X.append(output.cpu().detach().numpy())
        y[i] = carry
    return np.array(X), y

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X_train, y_train = data_probing(train_size)
X_test, y_test = data_probing(test_size)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

reg = LogisticRegression()
reg.fit(X_train,y_train)
reg.score(X_test, y_test)

0.99